# Notebook 11 — Network Medicine and Disease Modules

**Module:** 12 — Systems and Network Biology  
**Tier:** 2 — Working competence  
**Notebook:** 11 of 12  
**Time estimate:** 75 minutes

> A disease gene doesn't act alone. It acts within a module — a connected subnetwork
> of the interactome. Network medicine asks: where does the disease module sit,
> how large is it, and what does its neighborhood tell us about drug targets?

---
## Step 1 — Motivation

Genome-wide association studies (GWAS) identify hundreds of disease-associated
variants. Most encode proteins that interact directly or indirectly in the PPI network.
Network medicine (Barabási et al. 2011) formalizes this: the **disease module
hypothesis** states that disease proteins form a connected subnetwork in the PPI.
This has practical consequences for drug target identification.

---
## Step 2 — Intuition

**Disease module:** The set of all proteins associated with a disease, plus the
shortest paths connecting them in the PPI network — a connected "disease neighborhood."

**Drug-target proximity:** Drugs don't need to hit a disease protein directly;
they can hit proteins near the disease module. A drug target that is close to the
disease module (small network distance) is more likely to be effective.

**Module separation:** Two diseases with modules far apart in the network tend to have
distinct biological mechanisms. Two diseases with overlapping modules often share
comorbidities — this is the network basis of comorbidity.

**Mendelian disease proteins:** Tend to be in the periphery (low degree, module-specific).
**Complex disease proteins:** Tend to be hub proteins shared across many modules.

---
## Step 3 — Biological Background

**Key observations from Menche et al. (2015):**
- 70% of disease gene sets form a significant connected module in the human PPI
- Module size correlates with disease severity and genetic complexity
- Module overlap is a better predictor of comorbidity than gene set overlap alone

**Network proximity for drug-target validation (Guney et al. 2016):**
$$d(S, T) = \frac{1}{|S| + |T|} \left( \sum_{s \in S} \min_{t \in T} d(s,t) + \sum_{t \in T} \min_{s \in S} d(s,t) \right)$$

where $S$ = drug targets, $T$ = disease proteins.
Drugs with small $d(S,T)$ (targets close to disease module) more often work in trials.

**Protein complex diseases:**
- Protein complex members are typically co-essential and co-regulated
- Mutations disrupting complex stoichiometry cause "stoichiometric imbalance" diseases
  (e.g., trisomy 21: extra chromosome disrupts complex stoichiometry)

**Drug repurposing:** Find approved drugs whose target set is close to a new disease
module — candidate repurposed drug. Network proximity has been used to identify
COVID-19 treatment candidates using the SARS-CoV-2 human protein interaction network.

**Significance testing — random module null model:**
Compare observed module LCC size to random gene sets of the same size (degree-matched)
drawn from the same PPI. Z-score: $(\text{LCC}_{obs} - \mu_{rand}) / \sigma_{rand}$.

---
## Step 4 — Mathematical Explanation

**Largest connected component (LCC) of disease gene set $D$:**
$$\text{LCC}(D) = \max_{C \in \text{CC}(G[D])} |C|$$

where $G[D]$ is the subgraph induced by $D$ in the PPI $G$.

**Module significance Z-score:**
$$Z = \frac{\text{LCC}(D) - \mu_{\text{rand}}}{\sigma_{\text{rand}}}$$

$\mu_{\text{rand}}, \sigma_{\text{rand}}$ from 1000 random gene sets of size $|D|$
with matching degree distribution (degree-matched permutation).

$Z > 1.96$ → disease module is significantly more connected than random.

**Module-module separation:**
$$d_{AB} = \langle d(a,b) \rangle_{a \in A, b \in B} - \frac{1}{2}(d_{AA} + d_{BB})$$

where $d_{AA} = \langle d(a,a') \rangle$ within module $A$.
$d_{AB} > 0$: modules are separated; $d_{AB} < 0$: modules overlap.

In [ ]:
# Step 6 — Python: Network medicine analysis on synthetic PPI
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from collections import defaultdict

# ---- Generate synthetic PPI (re-use from NB07) ----
def generate_synthetic_ppi(n_per_module=60, n_modules=4, p_in=0.15, p_out=0.01,
                              n_extra_hubs=10, seed=42):
    rng = np.random.default_rng(seed)
    n = n_per_module * n_modules
    adj = {i: set() for i in range(n)}
    true_labels = np.repeat(np.arange(n_modules), n_per_module)
    for i in range(n):
        for j in range(i+1, n):
            p = p_in if true_labels[i] == true_labels[j] else p_out
            if rng.random() < p:
                adj[i].add(j); adj[j].add(i)
    for h in range(n_extra_hubs):
        hub_id = n + h
        adj[hub_id] = set()
        true_labels = np.append(true_labels, -1)
        degrees = np.array([len(adj[v]) for v in range(n)])
        probs = (degrees + 1.0) / (degrees + 1.0).sum()
        targets = rng.choice(n, size=min(25, n), replace=False, p=probs)
        for t in targets:
            adj[hub_id].add(t); adj[t].add(hub_id)
    n_total = n + n_extra_hubs
    return adj, true_labels, n_total

adj_ppi, true_labels, n_total = generate_synthetic_ppi()
G = nx.Graph()
for v in adj_ppi:
    for u in adj_ppi[v]:
        if u > v: G.add_edge(v, u)
lcc_nodes = max(nx.connected_components(G), key=len)
G_lcc = G.subgraph(lcc_nodes).copy()
print(f'PPI LCC: {G_lcc.number_of_nodes()} nodes, {G_lcc.number_of_edges()} edges')

# ---- Simulate disease gene sets from each module ----
# Disease A: 15 genes from module 0 (Block 0: nodes 0-59)
# Disease B: 15 genes from module 1 (Block 1: nodes 60-119)
# Disease C: 8 genes from module 0 AND 8 from module 1 (overlapping)
rng = np.random.default_rng(99)
module0 = [v for v in G_lcc.nodes() if true_labels[v] == 0]
module1 = [v for v in G_lcc.nodes() if true_labels[v] == 1]

disease_A = list(rng.choice(module0, size=15, replace=False))
disease_B = list(rng.choice(module1, size=15, replace=False))
disease_C = list(rng.choice(module0, size=8, replace=False)) + list(rng.choice(module1, size=8, replace=False))

def lcc_size(G, nodes):
    """LCC size of the induced subgraph G[nodes]."""
    nodes = [v for v in nodes if v in G]
    if not nodes: return 0
    sub = G.subgraph(nodes)
    if not sub.edges(): return 1
    return max(len(c) for c in nx.connected_components(sub))

def module_zscore(G, disease_genes, n_random=1000, seed=0):
    """Z-score of LCC size vs. random gene sets of same size."""
    rng = np.random.default_rng(seed)
    obs_lcc = lcc_size(G, disease_genes)
    all_nodes = list(G.nodes())
    n = len(disease_genes)
    rand_lccs = []
    for _ in range(n_random):
        rand_genes = rng.choice(all_nodes, size=n, replace=False)
        rand_lccs.append(lcc_size(G, rand_genes))
    mu, sigma = np.mean(rand_lccs), np.std(rand_lccs)
    z = (obs_lcc - mu) / max(sigma, 1e-6)
    return obs_lcc, mu, sigma, z, rand_lccs

for name, genes in [('Disease A', disease_A), ('Disease B', disease_B), ('Disease C', disease_C)]:
    obs, mu, sigma, z, _ = module_zscore(G_lcc, genes, n_random=500)
    print(f'{name}: LCC={obs}, random mean={mu:.1f}±{sigma:.1f}, Z={z:.2f}')

# ---- Drug-target proximity ----
def network_proximity(G, source_nodes, target_nodes):
    """Mean shortest path distance from source to nearest target (and vice versa)."""
    sources = [v for v in source_nodes if v in G]
    targets = [v for v in target_nodes if v in G]
    if not sources or not targets: return np.inf
    # One direction: source -> nearest target
    d_st = []
    for s in sources:
        dists = nx.single_source_shortest_path_length(G, s)
        min_d = min((dists.get(t, np.inf) for t in targets), default=np.inf)
        if min_d < np.inf: d_st.append(min_d)
    # Other direction: target -> nearest source
    d_ts = []
    for t in targets:
        dists = nx.single_source_shortest_path_length(G, t)
        min_d = min((dists.get(s, np.inf) for s in sources), default=np.inf)
        if min_d < np.inf: d_ts.append(min_d)
    if not d_st or not d_ts: return np.inf
    return (np.mean(d_st) + np.mean(d_ts)) / 2

# Simulate two drug target sets: Drug X near Disease A, Drug Y far
drug_X = list(rng.choice(module0, size=5, replace=False))  # targets in module 0 (near Disease A)
drug_Y = list(rng.choice(module1, size=5, replace=False))  # targets in module 1
print(f'\nDrug X proximity to Disease A: {network_proximity(G_lcc, drug_X, disease_A):.3f}')
print(f'Drug Y proximity to Disease A: {network_proximity(G_lcc, drug_Y, disease_A):.3f}')
print(f'(Smaller distance = more proximate = more likely to be effective)')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel A: PPI with disease modules highlighted
try:
    pos = nx.spring_layout(G_lcc, seed=42, k=0.3)
except Exception:
    pos = nx.circular_layout(G_lcc)

ax = axes[0]
nx.draw_networkx_edges(G_lcc, pos, ax=ax, alpha=0.15, width=0.3, edge_color='grey')

# Color: Disease A=red, Disease B=blue, Disease C=green, other=lightgrey
node_colors = []
for v in G_lcc.nodes():
    if v in disease_A: node_colors.append('tomato')
    elif v in disease_B: node_colors.append('steelblue')
    else: node_colors.append('lightgrey')

nx.draw_networkx_nodes(G_lcc, pos, ax=ax, node_color=node_colors,
                          node_size=[40 if v in disease_A or v in disease_B else 8 for v in G_lcc.nodes()],
                          alpha=0.8)
ax.set_title('A. Disease modules in PPI\n(red=Disease A, blue=Disease B)')
ax.axis('off')

# Panel B: LCC Z-score null distribution
ax = axes[1]
_, _, _, z_A, rand_A = module_zscore(G_lcc, disease_A, n_random=500)
ax.hist(rand_A, bins=20, color='lightgrey', edgecolor='black', density=True, label='Random')
obs_A = lcc_size(G_lcc, disease_A)
ax.axvline(obs_A, color='tomato', lw=2, label=f'Disease A LCC={obs_A}')
ax.set_xlabel('LCC size of random gene set')
ax.set_ylabel('Density')
ax.set_title(f'B. Module significance test\nDisease A: Z={z_A:.2f}')
ax.legend(fontsize=8)

# Panel C: Drug-target proximity landscape
ax = axes[2]
# Compute proximity for 50 random drug sets
rand_prox = []
for _ in range(80):
    rand_drug = list(rng.choice(list(G_lcc.nodes()), size=5, replace=False))
    p = network_proximity(G_lcc, rand_drug, disease_A)
    if p < np.inf: rand_prox.append(p)
ax.hist(rand_prox, bins=15, color='lightgrey', edgecolor='black', density=True, label='Random drugs')
ax.axvline(network_proximity(G_lcc, drug_X, disease_A), color='green', lw=2, label='Drug X (proximate)')
ax.axvline(network_proximity(G_lcc, drug_Y, disease_A), color='tomato', lw=2, label='Drug Y (distal)')
ax.set_xlabel('Network proximity to Disease A')
ax.set_ylabel('Density')
ax.set_title('C. Drug-target network proximity\n(lower = more proximate = more likely effective)')
ax.legend(fontsize=7)

plt.suptitle('Module 12 NB11: Network Medicine and Disease Modules',
               fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('network_medicine.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Compute the LCC Z-score for Disease C (the overlapping set). Is it more or less
   significant than Disease A alone?
2. Compute the module-module separation $d_{AB}$ between Disease A and Disease B
   modules. Are they separated or overlapping?
3. What would you predict about the comorbidity risk between Disease A and Disease B
   based on module separation?
4. Implement degree-matched random permutation: instead of drawing random genes
   uniformly, draw genes with the same degree distribution as the disease set.
   Does this change the Z-score significantly?

---
## Step 10 — Quiz

1. What is the disease module hypothesis?
2. How is module significance tested statistically?
3. What does drug-target network proximity measure?
4. How does module overlap relate to disease comorbidity?
5. What is degree-matched randomization and why is it important for module significance tests?

---
## Step 12 — Reflection

> *[In one paragraph: explain why two diseases with overlapping network modules are
> likely to show comorbidity, using the biological interpretation of what it means
> for disease proteins to be connected in the PPI.]*

---
*Next: `12_mini_project_and_assessment.ipynb`*